#### Sample script for reading in TC-DROPS data
Brian Tang

<i> 6 November 2025 </i>

In [ ]:
#Libraries

import re
import numpy as np
import xarray as xr
import pandas as pd
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
from metpy.calc import dewpoint_from_relative_humidity
from metpy.plots import SkewT
from metpy.units import units
import seaborn as sns
sns.set_context("paper",font_scale=1.0)

%matplotlib inline

In [ ]:
#open TC-DROPS dataset (we'll open the data on height levels)
ds_tcdrops = xr.open_dataset('/network/rit/lab/tcdynasty/tang/tcdrops/all_storms_height_dropsonde_dataset_ver1.2_updated.nc')

#print dataset info
ds_tcdrops

In [ ]:
#Specify storm and mission(s) of interest
atcfid = 'AL062014' #ATCF identifier
tcdropsid = 'al_edouard2014' #TCDROPS identifier <basin>_<name><year>
storm = re.search(r'_(\D+)\d+', tcdropsid).group(1) #extract just the storm name

#print all available missions for storm
stormname = np.squeeze(ds_tcdrops['storm_name'].values).astype(str)
ind = np.nonzero(stormname == tcdropsid)[0]
flightids = np.squeeze(ds_tcdrops['flight_id'].values).astype(str)
print(np.unique(flightids[ind]))

#specify missions of interest that you want to extract dropsonde data for
flightid = ['20140911I1', '20140911_hs3_gh','20140914I1', '20140914_hs3_gh', '20140915H1', '20140915I1', '20140915N1', '20140916H1', '20140916I1', '20140916_hs3_gh', '20140917H1', '20140918_hs3_gh', '20140919I1'] #can list one mission; group missions together to examine combined dropsondes

In [ ]:
#Open IBTracs to get TC track observations
ds_ibtracs = xr.open_dataset('/network/rit/lab/tcdynasty/tang/ibtracs/IBTrACS.since1980.v04r01.nc')

#determine index of TC
atcf = np.squeeze(ds_ibtracs['usa_atcf_id'][:,0].values).astype(str)
atcfind = np.nonzero(atcf == atcfid)[0][0]
print(atcfind)

#get track information of TC
ds_ibtracs_tc = ds_ibtracs[['usa_lat','usa_lon']].isel(storm = atcfind)

#convert to dataframe
df_ibtracs_tc = ds_ibtracs_tc.to_dataframe().dropna(how='all').reset_index()
df_ibtracs_tc = df_ibtracs_tc.drop(columns=['date_time','usa_lat','usa_lon'])
df_ibtracs_tc['time'] = df_ibtracs_tc['time'].dt.round('min')

#resample to every minute and apply cubic interpolation
df_ibtracs_tc = df_ibtracs_tc.set_index('time').resample('1s').interpolate(method='cubic')
df_ibtracs_tc = df_ibtracs_tc.reset_index()

# calculate storm motion
REarth = 6371.8E3
time_array = (df_ibtracs_tc['time']-df_ibtracs_tc['time'].iloc[0])/np.timedelta64(1,'s')
# zonal motion = d(lon)/dt * cos(lat) * Radius of Earth
df_ibtracs_tc['storm_motion_u'] = np.gradient(np.radians(df_ibtracs_tc.lon.values),time_array.values)*REarth*np.cos(np.radians(df_ibtracs_tc.lat.values))
# meridional motion = d(lat)/dt * Radius of Earth
df_ibtracs_tc['storm_motion_v'] = np.gradient(np.radians(df_ibtracs_tc.lat.values),time_array.values)*REarth
df_ibtracs_tc

In [ ]:
#Extract dropsonde data for specified mission(s) of interest
#subset mission(s) of interest
ind = []
for i in range(len(flightid)):
    ind.append(np.nonzero(flightids == flightid[i])[0])
ind = np.hstack(ind)

#get the launch time for each drop
drops = ds_tcdrops[['year','month','day','hour','minute','second','latitude_initial','longitude_initial',\
                       'lat_dist_initial','lon_dist_initial','storm_name','flight_id']].isel(ndrop = ind)
df_mission = drops.to_dataframe()
df_mission['storm_name'] = df_mission['storm_name'].str.decode("utf-8")
df_mission['flight_id'] = df_mission['flight_id'].str.decode("utf-8")

#convert launch date/time information to datetime
df_mission['launchtime'] = df_mission.apply(lambda row: datetime(int(row['year']),int(row['month']),int(row['day']),\
                                        int(row['hour']),int(row['minute']),int(row['second'])) ,axis=1)

#get all drops for mission
ds_tcdrop = ds_tcdrops[['time_elapsed','longitude','latitude','pressure','height','temperature','wvmr','rh','uwind','vwind','theta_e']].isel(ndrop = ind)

#convert to dataframe and clean up
df_tcdrop = ds_tcdrop.to_dataframe().dropna(how='all').reset_index()

#calculate time of observation
df_tcdrop = pd.merge(df_tcdrop,df_mission['launchtime'],on='ndrop')
df_tcdrop['time'] = df_tcdrop['launchtime'] + pd.to_timedelta(df_tcdrop['time_elapsed'], unit='s')
df_tcdrop = df_tcdrop.drop(columns=['time_elapsed','nlevels','launchtime'])

#merge with resampled IBTracs data
df_tcdrop['time'] = df_tcdrop['time'].dt.round('s') #round time in TC-DROPS to nearest second
df_tcdrop = pd.merge(df_tcdrop,df_ibtracs_tc,on='time')

#rename TC position columns to avoid confusion
df_tcdrop = df_tcdrop.rename(columns={'lat': 'tclat', 'lon': 'tclon'})

df_tcdrop

In [ ]:
#Perform calculations to transform variables into cylindrical grid and storm-relative frame of reference
#calculate x and y distance relative to center, along with radius and azimuth
tcphi = df_tcdrop.tclat*np.pi/180
tclam = df_tcdrop.tclon*np.pi/180
phi = df_tcdrop.latitude*np.pi/180
lam = df_tcdrop.longitude*np.pi/180

#transformation variables to account for sphericity of lat/lon grid, following Ahern and Cowan (2018)
a = np.clip(np.sin(tcphi)*np.sin(phi)+np.cos(tcphi)*np.cos(phi)*np.cos(phi-tcphi),-1,1)
ctrangle = np.maximum(np.arccos(a),1E-3)
df_tcdrop['x_dist'] = ctrangle*(REarth+df_tcdrop.height)/np.sin(ctrangle)*np.cos(phi)*np.sin(lam-tclam)
df_tcdrop['y_dist'] = ctrangle*(REarth+df_tcdrop.height)/np.sin(ctrangle)*(np.cos(tcphi)*np.sin(phi)-\
                                np.sin(tcphi)*np.cos(phi)*np.cos(lam-tclam))
df_tcdrop['radius'] = (df_tcdrop.x_dist**2.0+df_tcdrop.y_dist**2.0)**0.5
theta_xy = np.arctan2(df_tcdrop.y_dist,df_tcdrop.x_dist)
df_tcdrop['azimuth'] = np.where(theta_xy>=0,theta_xy,theta_xy+2*np.pi)

In [ ]:
#Calculate storm-relative u, v, radial, and tangential wind, following Ahern and Cowan (2018)
df_tcdrop['u_wind_relative']=df_tcdrop.uwind-df_tcdrop.storm_motion_u
df_tcdrop['v_wind_relative']=df_tcdrop.vwind-df_tcdrop.storm_motion_v

eps = 1.E-3
wspd = np.sqrt(df_tcdrop.u_wind_relative**2+df_tcdrop.v_wind_relative**2)
lamprime = lam + eps*df_tcdrop.u_wind_relative/(wspd*np.cos(phi))
phiprime = phi + eps*df_tcdrop.v_wind_relative/wspd

b = np.clip(np.sin(tcphi)*np.sin(phiprime)+np.cos(tcphi)*np.cos(phiprime)*np.cos(phiprime-tcphi),-1,1)
ctrangleprime = np.maximum(np.arccos(b),1E-3)
xprime = ctrangleprime*(REarth+df_tcdrop.height)/np.sin(ctrangleprime)*np.cos(phiprime)*np.sin(lamprime-tclam)
yprime = ctrangleprime*(REarth+df_tcdrop.height)/np.sin(ctrangleprime)*(np.cos(tcphi)*np.sin(phiprime)-\
                        np.sin(tcphi)*np.cos(phiprime)*np.cos(lamprime-tclam))
alpha = np.arctan2(yprime-df_tcdrop.y_dist,xprime-df_tcdrop.x_dist)
alpha = np.where(alpha>=0,alpha,alpha+2*np.pi)

df_tcdrop['radwind_relative'] = wspd*np.cos(df_tcdrop.azimuth-alpha)
df_tcdrop['tanwind_relative'] = wspd*np.cos(df_tcdrop.azimuth-alpha+np.pi/2)

In [ ]:
#Get rid of NaNs and sort
df_tcdrop = df_tcdrop.set_index('time').groupby('ndrop', group_keys=True).apply(lambda group : group.interpolate(method='time',limit_area='inside'))
df_tcdrop = df_tcdrop.drop(columns=['ndrop'])
df_tcdrop = df_tcdrop.dropna().sort_values(['ndrop','time'],ascending=[True,False]).reset_index()

df_tcdrop

#### Examples diagnostics and visualizations

In [ ]:
#Plot dropsonde points in TC-relative frame of reference
#group into drops from low-level and high-level aircraft
pltlim = 400
df_tcdrop_release = df_tcdrop.groupby('ndrop').tail(1)
df_tcdrop_release_hi = df_tcdrop_release[df_tcdrop_release['pressure']<500] #low-level recon drops
df_tcdrop_release_lo = df_tcdrop_release[df_tcdrop_release['pressure']>=500] #high-level recon drops

#plot release points (note: only plots points within axes range)
fig = plt.figure(figsize=(8,8))
plt.plot(df_tcdrop_release_hi['x_dist']/1E3,df_tcdrop_release_hi['y_dist']/1E3,'x',color='red',ms=7,mew=1.5)
plt.plot(df_tcdrop_release_lo['x_dist']/1E3,df_tcdrop_release_lo['y_dist']/1E3,'+',color='blue',ms=8,mew=1.5)
for j in range(len(df_tcdrop_release)):
    plt.text(df_tcdrop_release['x_dist'].iloc[j]/1E3,df_tcdrop_release['y_dist'].iloc[j]/1E3,df_tcdrop_release['ndrop'].iloc[j],clip_on=True)
plt.ylim(-pltlim,pltlim)
plt.xlim(-pltlim,pltlim)

#range rings
crad = np.arange(100,401,100)
for i in range(len(crad)):
    c = plt.Circle((0,0), radius=crad[i], edgecolor='gray', facecolor='none', linestyle='--', linewidth=0.5)
    plt.gca().add_artist(c)
plt.plot([-400,400],[0,0],color='gray',linestyle='--', linewidth=0.5)
plt.plot([0,0],[-400,400],color='gray',linestyle='--', linewidth=0.5)

plt.xlabel('zonal distance from center (km)')
plt.ylabel('meridional distance from center (km)')
plt.show()

In [ ]:
#Plot skew-T and ventilation profile
#select dropsonde to plot
sid = 78 #isolate 

fig = plt.figure(figsize=(8,8))
skew = SkewT(fig, rotation=35, aspect=90)

df_sonde = df_tcdrop[(df_tcdrop['ndrop'] == sid)].sort_values(by=['pressure'])
p = df_sonde['pressure'].values * units.hPa
T = df_sonde['temperature'].values * units.degC
u = df_sonde['uwind'].values * units('m/s')
v = df_sonde['vwind'].values * units('m/s')
Td = dewpoint_from_relative_humidity(df_sonde['temperature'].values * units.degC, df_sonde['rh'].values * units.percent)
skew.plot(p, T, lw=2.5, color='tab:red')
skew.plot(p, Td,lw=2.5, color='tab:green')
skew.plot_barbs(p[::10], u[::10], v[::10], length=6)

skew.ax.set_ylim(1012, 200)
skew.ax.set_xlim(-25, 30)
skew.ax.set_xlabel('Temperature, dewpoint (C)')
skew.ax.set_ylabel('Pressure (hPa)')

skew.plot_dry_adiabats(lw=0.5)
skew.plot_moist_adiabats(lw=0.5)
skew.plot_mixing_lines(lw=0.5)

plt.show()

In [ ]:
###########################################          My code here

In [ ]:
#---------------------------------------------------------------------------------------------------------------------------

In [ ]:
RMW_KM = 27.78
TARGET_TIME = "2023-09-15 05:00:00"
TARGET_QUAD = "NE"


In [ ]:
import xarray as xr

ds_tcdrops = xr.open_dataset(
    "/network/rit/lab/tcdynasty/tang/tcdrops/all_storms_height_dropsonde_dataset_ver1.2_updated.nc"
)

print("Dataset opened successfully")
print(ds_tcdrops)


In [ ]:
print("Key variables needed:")
for v in ["dtg", "drop_quad", "rel_dist", "stat_rmax", "vertical_levels"]:
    print(v, "→", v in ds_tcdrops.variables)


In [ ]:
unique_times = np.unique(ds_tcdrops["dtg"].values)

# Print times within ±2 hours of 05Z
for t in unique_times:
    if np.datetime64("2014-09-15T04:00") <= t <= np.datetime64("2014-09-15T07:00"):
        print(t)


In [ ]:
import numpy as np

# Correct year and target time
TARGET_TIME = np.datetime64("2014-09-15T05:00")
TARGET_QUAD = "FR"

# Allow ±30 minutes around target time
time_tol = np.timedelta64(30, 'm')

mask = (
    (np.abs(ds_tcdrops["dtg"] - TARGET_TIME) <= time_tol) &
    (ds_tcdrops["drop_quad"] == TARGET_QUAD)
)

print("Number of True drops:", mask.sum().values)


In [ ]:
unique_times = np.unique(ds_tcdrops["dtg"].values)
for t in unique_times:
    if np.datetime64("2014-09-15T04:00") <= t <= np.datetime64("2014-09-15T07:00"):
        print(t)


In [ ]:
import numpy as np

# Use a ±30 min window around 05Z
TARGET_TIME = np.datetime64("2014-09-15T05:00")
time_tol = np.timedelta64(30, "m")  # ±30 minutes
TARGET_QUAD = "FR"

mask = (
    (np.abs(ds_tcdrops["dtg"] - TARGET_TIME) <= time_tol) &
    (ds_tcdrops["drop_quad"] == TARGET_QUAD)
)

print("Number of drops selected:", mask.sum().values)

# Extract the drops
drops = ds_tcdrops.isel(ndrop=mask)

# Sort by distance from storm center for cross section
sort_idx = np.argsort(drops["rel_dist"])
drops_sorted = drops.isel(ndrop=sort_idx)

print("First few relative distances (km):", drops_sorted["rel_dist"].values[:10])


In [ ]:
print("Number of NE (FR) drops:", mask.sum().values)


In [ ]:
import numpy as np

# Filter for Edouard 2014 drops
tcdropsid = 'al_edouard2014'
edouard_mask = (ds_tcdrops["storm_name"].values.astype(str) == tcdropsid)

# Only consider these drops
ds_edouard = ds_tcdrops.isel(ndrop=edouard_mask)

# Quadrant of interest (FR → NE)
TARGET_QUAD = "NE"

# Loop through unique times in Edouard drops
unique_times = np.unique(ds_edouard["dtg"].values)

print("Times with NE (FR) drops in Edouard 2014:")
for t in unique_times:
    mask = (ds_edouard["dtg"] == t) & (ds_edouard["drop_quad"] == TARGET_QUAD)
    n_drops = mask.sum().values
    if n_drops > 0:
        print(f"{t} → {n_drops} drops")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Flight ID to analyze (replace with the flight you want)
FLIGHT_ID = "20140914I1"  # example
ds_flight_mask = (ds_edouard["flight_id"].values.astype(str) == FLIGHT_ID)

ds_flight = ds_edouard.isel(ndrop=ds_flight_mask)
print(f"Number of drops in this flight: {ds_flight['ndrop'].size}")


In [ ]:
rel_dist = ds_flight["rel_dist"].values  # shape: (ndrops,)
vertical_levels = ds_flight["vertical_levels"].values  # shape: (nlevels,)

# Example: use theta_e if available
if "theta_e" in ds_flight.variables:
    var = ds_flight["theta_e"].values  # shape: (ndrops, nlevels)
elif "temp" in ds_flight.variables:
    var = ds_flight["temp"].values
else:
    var = np.random.rand(len(rel_dist), len(vertical_levels))  # placeholder


In [ ]:
n_drops = rel_dist.shape[0]
n_levels = vertical_levels.shape[0]

# Repeat distances and vertical levels
x = np.repeat(rel_dist, n_levels)        # each drop distance repeated for all vertical levels
y = np.tile(vertical_levels, n_drops)    # vertical levels repeated for each drop
c = var.flatten()                        # variable values for coloring


In [ ]:
#------------------------------------------------------------------------------------------------

In [ ]:
print(ds_edouard)
print("\nVariables:")
for v in ds_edouard.data_vars:
    print(" -", v)

print("\nCoordinates:")
for c in ds_edouard.coords:
    print(" -", c)


In [ ]:
import numpy as np

t0 = np.datetime64("2014-09-14T22:00")
t1 = np.datetime64("2014-09-15T00:00")

mask_time = (ds_edouard["dtg"] >= t0) & (ds_edouard["dtg"] <= t1)

ds_pass = ds_edouard.isel(ndrop=mask_time)

print("Number of drops in pass:", ds_pass.sizes["ndrop"])
print("Times in pass:")
print(np.unique(ds_pass["dtg"].values))


In [ ]:
x = np.arange(ds_pass.sizes["ndrop"])
print("Cross-section length:", len(x))


In [ ]:
z = ds_pass["pressure"].values  # (ndrop, nlevels)
print("z shape:", z.shape)


In [ ]:
vspeed = ds_pass["vspeed"].values     # vertical velocity
theta_e = ds_pass["theta_e"].values
rh = ds_pass["rh"].values
temperature = ds_pass["temperature"].values

print("vspeed:", vspeed.shape)
print("theta_e:", theta_e.shape)


In [ ]:
import matplotlib.pyplot as plt

# pick first 3 drops
for i in range(min(3, Z.shape[0])):
    plt.plot(Z[i,:], y, label=f"Drop {i} at {x[i]:.1f} km")

plt.gca().invert_yaxis()
plt.xlabel("Theta_e (K)")
plt.ylabel("Pressure (hPa)")
plt.title("Vertical profiles for first few drops")
plt.legend()
plt.show()


In [ ]:
print("Number of NaNs in vspeed:", np.sum(np.isnan(vspeed)))
print("Total vspeed values:", vspeed.size)


In [ ]:
print("Mean theta_e by drop (ignoring NaNs):")
for i in range(theta_e.shape[1]):
    prof_mean = np.nanmean(theta_e[:, i])
    print(f"Drop {i}: mean theta_e = {prof_mean:.2f} K")


In [ ]:
best_track_data = [
    ("10/1800", 13.7, -33.2, 1009, 25, "low"),
    ("11/0000", 14.2, -34.4, 1009, 25, "low"),
    ("11/0600", 14.9, -35.5, 1009, 25, "low"),
    ("11/1200", 15.8, -36.5, 1008, 30, "tropical depression"),
    ("11/1800", 16.5, -37.7, 1007, 30, "tropical depression"),
    ("12/0000", 17.2, -39.2, 1006, 35, "tropical storm"),
    ("12/0600", 17.7, -40.8, 1005, 35, "tropical storm"),
    ("12/1200", 18.3, -42.3, 1000, 40, "tropical storm"),
    ("12/1800", 18.9, -43.6, 999, 40, "tropical storm"),
    ("13/0000", 19.6, -44.8, 998, 40, "tropical storm"),
    ("13/0600", 20.4, -45.7, 997, 45, "tropical storm"),
    ("13/1200", 21.4, -46.4, 996, 45, "tropical storm"),
    ("13/1800", 22.3, -47.0, 995, 50, "tropical storm"),
    ("14/0000", 23.0, -47.9, 994, 55, "hurricane"),
    ("14/0600", 23.7, -49.0, 991, 60, "hurricane"),
    ("14/1200", 24.4, -50.3, 985, 70, "hurricane"),
    ("14/1800", 25.1, -51.5, 980, 75, "hurricane"),
    ("15/0000", 25.8, -52.7, 975, 80, "hurricane"),
    ("15/0600", 26.5, -54.0, 970, 85, "hurricane"),
    ("15/1200", 27.0, -55.1, 966, 90, "hurricane"),
    ("15/1800", 27.7, -56.1, 962, 95, "hurricane"),
    ("16/0000", 28.5, -56.7, 960, 95, "hurricane"),
    ("16/0600", 29.5, -57.3, 956, 100, "hurricane"),
    ("16/1200", 30.6, -57.8, 955, 105, "hurricane"),
    ("16/1800", 31.7, -57.7, 958, 95, "hurricane"),
    ("17/0000", 32.9, -57.0, 959, 85, "hurricane"),
    ("17/0600", 34.3, -56.0, 958, 85, "hurricane"),
    ("17/1200", 35.7, -54.5, 957, 80, "hurricane"),
    ("17/1800", 37.2, -52.4, 955, 80, "hurricane"),
    ("18/0000", 38.3, -49.5, 958, 75, "hurricane"),
    ("18/0600", 39.3, -46.7, 964, 70, "hurricane"),
    ("18/1200", 39.9, -44.1, 969, 70, "hurricane"),
    ("18/1800", 39.9, -42.1, 973, 65, "hurricane"),
    ("19/0000", 39.9, -40.1, 980, 60, "tropical storm"),
    ("19/0600", 39.9, -38.9, 988, 55, "tropical storm"),
    ("19/1200", 39.5, -38.1, 994, 45, "tropical storm"),
    ("19/1800", 39.5, -37.4, 1001, 40, "low"),
    ("20/0000", 39.4, -36.6, 1004, 35, "low"),
    ("20/0600", 39.3, -35.7, 1006, 30, "low"),
    ("20/1200", 39.1, -34.4, 1008, 30, "low"),
    ("20/1800", 38.8, -33.0, 1010, 25, "low"),
    ("21/0000", 38.1, -31.8, 1012, 25, "low"),
]

In [ ]:
import pandas as pd

bt = pd.DataFrame(
    best_track_data,
    columns=["time", "lat", "lon", "mslp", "vmax", "status"]
)

print(bt.head())


In [ ]:
center_row = bt[bt["time"] == "15/0600"].iloc[0]

center_lat = center_row["lat"]
center_lon = center_row["lon"]

print("Using storm center:")
print(center_row)


In [ ]:
lat = ds_tcdrops["latitude"].values    # shape (nlevels, ndrop)
lon = ds_tcdrops["longitude"].values   # shape (nlevels, ndrop)

print("lat shape:", lat.shape)
print("lon shape:", lon.shape)


In [ ]:
# Make sure we're only working on Edouard
ds_e = ds_tcdrops.where(
    ds_tcdrops.storm_name.astype(str) == "al_edouard2014",
    drop=True
)

# Time window: 2014-09-14 22–24 UTC
t0 = np.datetime64("2014-09-14T22:00")
t1 = np.datetime64("2014-09-15T00:00")

ds_e = ds_e.where(
    (ds_e.dtg >= t0) & (ds_e.dtg <= t1),
    drop=True
)

print("Drops after filtering:", ds_e.dims["ndrop"])


In [ ]:
lat = ds_e["latitude"].values     # (441, ndrop)
lon = ds_e["longitude"].values    # (441, ndrop)

print("lat shape:", lat.shape)
print("lon shape:", lon.shape)


In [ ]:
import numpy as np

# Median latitude/longitude across vertical levels for each drop
lats_drop = np.nanmedian(ds_e["latitude"].values, axis=0)   # shape: (ndrop,)
lons_drop = np.nanmedian(ds_e["longitude"].values, axis=0)  # shape: (ndrop,)
times_drop = ds_e["dtg"].values                              # shape: (ndrop,)

print(f"Drops after filtering: {len(times_drop)}\n")

# Print a simple table
for i in range(len(times_drop)):
    print(f"Drop {i+1}: Time = {times_drop[i]}, Latitude = {lats_drop[i]:.3f}, Longitude = {lons_drop[i]:.3f}")


In [ ]:
center_lat = 25.1
center_lon = -51.5


In [ ]:
R_earth_km = 6371.0

lat_rad = np.deg2rad(lat)
lon_rad = np.deg2rad(lon)
clat = np.deg2rad(center_lat)
clon = np.deg2rad(center_lon)

dlat = lat_rad - clat
dlon = lon_rad - clon

a = (
    np.sin(dlat/2)**2
    + np.cos(clat) * np.cos(lat_rad) * np.sin(dlon/2)**2
)

c = 2 * np.arcsin(np.sqrt(a))
dist_km = R_earth_km * c   # (441, ndrop)


In [ ]:
print("Distance min/max (km):",
      np.nanmin(dist_km),
      np.nanmax(dist_km))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Axes
x = np.nanmean(dist_km, axis=0)          # (ndrop,)
y = ds_e["vertical_levels"].values       # (nlevels,)
Z = ds_e["theta_e"].values               # (nlevels, ndrop)

# Mask columns where theta_e is all NaN
valid_levels = np.any(np.isfinite(Z), axis=1)

y = y[valid_levels]
Z = Z[valid_levels, :]

fig, ax = plt.subplots(figsize=(9, 6))

pcm = ax.pcolormesh(
    x,
    y,
    Z,
    shading="nearest",
    cmap="Spectral_r"
)

ax.set_xlabel("Radius from storm center (km)")
ax.set_ylabel("Height (m)")
ax.set_title("Theta-e cross section across outer rainband")

cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label("Equivalent potential temperature (K)")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Extract height and theta_e
height = ds_e["height"].values   # shape (nlevels, ndrops)
theta_e = ds_e["theta_e"].values # shape (nlevels, ndrops)

nlevels, ndrops = height.shape

# Build vertical axis: use median across drops, ignoring NaNs
y = np.nanmedian(height, axis=1)  # shape (nlevels,)
valid_levels = np.isfinite(y)     # only keep levels with some valid height

# Keep only valid levels
y = y[valid_levels]
Z_hov = theta_e[valid_levels, :]  # shape (n_valid_levels, ndrops)

# Optional: interpolate missing theta_e along vertical for smoother plot
for i in range(ndrops):
    valid = np.isfinite(Z_hov[:, i])
    if valid.sum() >= 2:
        Z_hov[:, i] = np.interp(np.arange(len(y)), np.flatnonzero(valid), Z_hov[valid, i])
    else:
        Z_hov[:, i] = np.nan

# Horizontal axis: distance along rainband (or drop index)
x = np.arange(ndrops)

# Plot Hovmöller
fig, ax = plt.subplots(figsize=(10,6))
pcm = ax.pcolormesh(x, y, Z_hov, shading='auto', cmap='Spectral_r')
#ax.invert_yaxis()  # higher heights at top
ax.set_xlabel("Drop index / Distance along rainband")
ax.set_ylabel("Height (m)")
cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label("θₑ (K)")
ax.set_title("Hovmöller Diagram of θₑ across Rainband")
plt.show()


In [ ]:
# Horizontal axis: distance from storm center (km)
x = np.nanmedian(dist_km, axis=0)  # shape: (ndrops,)


In [ ]:
# Plot Hovmöller with distance in km
fig, ax = plt.subplots(figsize=(10,6))
pcm = ax.pcolormesh(x, y, Z_hov, shading='auto', cmap='Spectral_r')
ax.set_xlabel("Distance from storm center (km)")
ax.set_ylabel("Height (m)")
cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label("θₑ (K)")
ax.set_title("Diagram of θₑ across Rainband")
plt.show()
